In [4]:
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:

from isometry_pursuit.transformation import exponential_transformation
from isometry_pursuit.algorithm import group_basis_pursuit

import random

np.random.seed(1234)
random.seed(1234)


In [50]:
# Simulation setup
D = 50 # 3  # Dimension
P = 100  # Total number of columns (P > D)
power = 1.0

# Create D×D identity matrix as first D columns
I_D = np.eye(D)

# Generate random unit vectors (length 1) in R^D for remaining columns
num_random_vectors = P - D
random_vectors = np.random.randn(D, num_random_vectors)
# Normalize to unit length
random_unit_vectors = random_vectors / np.linalg.norm(random_vectors, axis=0, keepdims=True)

# Concatenate: identity matrix first, then random unit vectors
X = np.hstack([I_D, random_unit_vectors])

print(f"Matrix X shape: {X.shape}")
print(f"First {D} columns (identity):\n{X[:, :D]}")
print(f"\nRemaining {num_random_vectors} columns (random unit vectors):\n{X[:, D:]}")
print(f"\nColumn norms:\n{np.linalg.norm(X, axis=0)}")


Matrix X shape: (50, 100)
First 50 columns (identity):
[[1. 0. 0. ... 0. 0. 0.]
 [0. 1. 0. ... 0. 0. 0.]
 [0. 0. 1. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 1. 0. 0.]
 [0. 0. 0. ... 0. 1. 0.]
 [0. 0. 0. ... 0. 0. 1.]]

Remaining 50 columns (random unit vectors):
[[ 0.16905391 -0.1072288  -0.04007991 ...  0.03001349  0.00531319
  -0.26036114]
 [-0.2276951   0.09621117  0.03389698 ... -0.04472458  0.04577195
  -0.0557165 ]
 [-0.14306201 -0.10282816 -0.10117057 ...  0.13770384  0.05476214
  -0.01778238]
 ...
 [-0.23336276  0.08773367  0.00542632 ... -0.17219604 -0.16126232
  -0.11631672]
 [ 0.0874525  -0.19681237  0.0082609  ...  0.10090101 -0.02301532
  -0.24254763]
 [ 0.17702774 -0.03608492  0.14462332 ...  0.09695998  0.04954291
  -0.05618151]]

Column norms:
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1.

In [51]:
# Apply exponential transformation
X_transformed = exponential_transformation(X, power=power)
print(f"Transformed matrix shape: {X_transformed.shape}")


Transformed matrix shape: (50, 100)


In [52]:
# Run isometry pursuit (group basis pursuit)
beta = group_basis_pursuit(X_transformed)
print(f"Beta shape: {beta.shape}")
print(f"Beta (first few rows):\n{beta[:5, :]}")


Beta shape: (100, 50)
Beta (first few rows):
[[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0.]]


In [53]:
# Extract support: indices where beta has non-zero rows
support = np.where(np.linalg.norm(beta, axis=1) > 1e-6)[0]

print(f"Support (selected column indices): {support}")
print(f"Number of selected columns: {len(support)}")
print(f"\nSelected columns are:")
for idx in support:
    if idx < D:
        print(f"  Column {idx}: Identity matrix column (standard basis vector)")
    else:
        print(f"  Column {idx}: Random unit vector")
print(f"\nExpected: Identity matrix columns (0 to {D-1}) should be selected")
print(f"Actual selection: {support}")


Support (selected column indices): [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49]
Number of selected columns: 50

Selected columns are:
  Column 0: Identity matrix column (standard basis vector)
  Column 1: Identity matrix column (standard basis vector)
  Column 2: Identity matrix column (standard basis vector)
  Column 3: Identity matrix column (standard basis vector)
  Column 4: Identity matrix column (standard basis vector)
  Column 5: Identity matrix column (standard basis vector)
  Column 6: Identity matrix column (standard basis vector)
  Column 7: Identity matrix column (standard basis vector)
  Column 8: Identity matrix column (standard basis vector)
  Column 9: Identity matrix column (standard basis vector)
  Column 10: Identity matrix column (standard basis vector)
  Column 11: Identity matrix column (standard basis vector)
  Column 12: Identity matrix column (standard b

In [ ]:
# Visualize the selected vectors (if D <= 3 for visualization)
if D <= 3:
    fig = plt.figure(figsize=(10, 8))
    
    if D == 2:
        ax = fig.add_subplot(111)
        # Plot all vectors
        for i in range(P):
            color = 'red' if i in support else 'gray'
            alpha = 1.0 if i in support else 0.3
            ax.quiver(0, 0, X[0, i], X[1, i], angles='xy', scale_units='xy', scale=1,
                     color=color, alpha=alpha, width=0.005)
            if i in support:
                ax.text(X[0, i]*1.1, X[1, i]*1.1, f'{i}', fontsize=10)
        
        # Add unit circle
        circle = plt.Circle((0, 0), 1, color='black', linestyle='dotted', fill=False, linewidth=1)
        ax.add_artist(circle)
        
        ax.set_xlim(-1.5, 1.5)
        ax.set_ylim(-1.5, 1.5)
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.3)
        ax.set_title('Selected Vectors (Red = Selected, Gray = Not Selected)')
        ax.set_xlabel('Dimension 1')
        ax.set_ylabel('Dimension 2')
    
    elif D == 3:
        ax = fig.add_subplot(111, projection='3d')
        # Plot all vectors
        for i in range(P):
            color = 'red' if i in support else 'gray'
            alpha = 1.0 if i in support else 0.3
            ax.quiver(0, 0, 0, X[0, i], X[1, i], X[2, i], 
                     color=color, alpha=alpha, arrow_length_ratio=0.1)
            if i in support:
                ax.text(X[0, i]*1.1, X[1, i]*1.1, X[2, i]*1.1, f'{i}', fontsize=8)
        
        ax.set_xlim(-1.5, 1.5)
        ax.set_ylim(-1.5, 1.5)
        ax.set_zlim(-1.5, 1.5)
        ax.set_xlabel('Dimension 1')
        ax.set_ylabel('Dimension 2')
        ax.set_zlabel('Dimension 3')
        ax.set_title('Selected Vectors (Red = Selected, Gray = Not Selected)')
    
    plt.tight_layout()
    plt.show()
else:
    print(f"Visualization skipped (D={D} > 3). Set D <= 3 to visualize.")
